# 05 · REVEL — a supervised meta-predictor & the circularity problem

**REVEL** (*Rare Exome Variant Ensemble Learner*, Ioannidis et al. 2016, *AJHG*, PMID 27666373) is a **supervised** random-forest **ensemble** of 13 predictors, trained on curated pathogenic/benign variants. That training is powerful — but it makes benchmarking REVEL against ClinVar **partly circular**.


> ✅ **REAL DATA.** Genome-wide **REVEL v1.3** for CFTR — **~9,730** variants (`data/revel_cftr_v1.3.csv`, built by a manual-download build cell below), scored under CFTR's canonical transcript. **Keyed by genomic coordinate** (REVEL has no protein position) — join onto observed variants by `chrom,pos,ref,alt`. **Non-commercial license.** `source == 'REAL'`.


In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline


## 1 · REVEL — what is it?

**REVEL** = *Rare Exome Variant Ensemble Learner* (Ioannidis et al. 2016, *American Journal of
Human Genetics*, **PMID 27666373**).

Think of REVEL as a **committee vote of 13 other predictors**. Instead of inventing a brand-new way
to judge a missense variant, its authors took the scores of **13 individual tools** (e.g. SIFT,
PolyPhen-2, MutationTaster, and several conservation scores) and trained a **random forest** — a
supervised machine-learning model — to combine them into one number.

Key facts to remember:

| Property | REVEL |
|---|---|
| Learning type | **Supervised** (learned from labelled examples) |
| Model | Random-forest **ensemble** of 13 component predictors |
| Trained on | Curated **pathogenic** and **benign** missense variants |
| Score range | **0 → 1** (higher = more likely pathogenic) |
| Covers | Missense variants only |

The "supervised" part is crucial: someone had to **hand REVEL a training set of variants already
labelled pathogenic or benign** — from databases like **ClinVar** and **HGMD**, which is the
source of REVEL's circularity problem: benchmarking REVEL against the same clinical databases it
partly learned from is not a fully independent test. (AlphaMissense, EVE, and ESM1b in tools/02–04
never saw those labels at all, which is why they're a fairer comparison against ClinVar.)


## Building the REAL data — a manual download (no API, no per-gene file)

REVEL has no API and no small per-gene download — only the **genome-wide table**,
~6.5 GB uncompressed, packaged in a zip. **You must fetch this one yourself:**

1. Go to <https://sites.google.com/site/revelgenomics> and download
   **`revel-v1.3_all_chromosomes.zip`**.
2. Save it as `data/revel-v1.3_all_chromosomes.zip` (gitignored — never commit it).

Inside the zip is a single ~6.5 GB CSV literally named `revel_with_transcript_ids`
— **no `.csv` extension**; that is just how the REVEL authors package it, not a
corrupt file. It is keyed by **genomic coordinate** (`chrom, hg19_pos, grch38_pos,
ref, alt, aaref, aaalt, REVEL, Ensembl_transcriptid`), with **no protein
position** — join on coordinate, not `protein_variant` (one protein change can
arise from several different codon changes).

The cell below **streams the CSV row-by-row without loading 6.5 GB into memory**:
it reads until it reaches chromosome 7, keeps rows inside the CFTR GRCh38 window,
and stops as soon as it leaves the contiguous chr7 block (the file is
chromosome-grouped, so this is safe and fast).

> ⚠️ **Build gotcha (not a strand gotcha):** CFTR is on the **plus strand**, so `ref`/`alt` are the same as the coding change and need no complementing — e.g. `R334W` (`c.1000C>T`) is simply `7-117540230-C-T`. What *does* silently match nothing is mixing **GRCh37 and GRCh38** coordinates — REVEL's raw table has both `hg19_pos` and `grch38_pos` columns, so pick the right one for your build (this repo uses `grch38_pos`, matching every other tool here).

**Canonical-transcript filtering.** CFTR has multiple annotated transcripts, and
REVEL's raw file tags each row with every transcript ID that shares that exact
amino-acid call at that genomic site — e.g. a row tagged
`ENST00000003084;ENST00000454343;ENST00000426809` means all three transcripts
agree on that substitution. In this window, one transcript never appears in a
shared tag: `ENST00000468795`, on 1,096 rows. Checked directly against **live
Ensembl VEP** (not REVEL's static 2021 file): **zero of those 1,096** rows match
what current Ensembl says that transcript actually does at those positions —
REVEL's tag for it is stale, and Ensembl itself flags `ENST00000468795` with
`cds_start_NF` ("CDS start not found") on every one of them, meaning even its
*current* annotation is a low-confidence, unconfirmed-start model. It isn't a
real second opinion about CFTR biology, so the build cell captures
`Ensembl_transcriptid` and `load_revel()` keeps only rows tagged with CFTR's
**canonical transcript** (`ENST00000003084`, matching MANE `NM_000492.4` — the
transcript every other join in this toolkit assumes). That drops the raw
10,826 rows to **9,730** canonical-scored sites; see the verification cell
below for the live count.

License: free for **non-commercial use** (contact the REVEL authors otherwise) —
see `data_manifest.json`.

**Version & reproducibility.** Same situation as EVE/ESM1b (tools/03–04): a
manual, authenticated-adjacent download, not something this cell fetches over
HTTP, so there's no `Last-Modified` header to grab. The zip is still versioned,
though — its one member, `revel_with_transcript_ids`, carries an embedded
timestamp from when the REVEL authors packaged it: **2021-05-03 15:50:38**,
readable straight from the zip's own metadata (`zipfile.ZipInfo.date_time`), no
download needed. The build cell records it into `data/revel_cftr_v1.3.release.json`;
`load_revel()` exposes it as the `revel_release` column, the same way
`eve_release` and `esm1b_release` work.


In [2]:
import zipfile, io, csv, json
from datetime import datetime

DATA_DIR = pathlib.Path.cwd().parent / "data"
REVEL_ZIP = DATA_DIR / "revel-v1.3_all_chromosomes.zip"
REVEL_MEMBER = "revel_with_transcript_ids"
REVEL_TSV = DATA_DIR / "revel_cftr_v1.3.csv"
REVEL_RELEASE_JSON = DATA_DIR / "revel_cftr_v1.3.release.json"
CFTR_WINDOW = (117_470_000, 117_670_000)   # GRCh38, safe superset of the CFTR locus

if REVEL_TSV.exists():
    print(f"already built -> {REVEL_TSV.name} (delete it and {REVEL_RELEASE_JSON.name} to rebuild)")
elif not REVEL_ZIP.exists():
    raise FileNotFoundError(
        f"{REVEL_ZIP} not found.\n"
        "REVEL has no API and no per-gene download -- get the genome-wide table:\n"
        "  1. Go to https://sites.google.com/site/revelgenomics and download\n"
        "     'revel-v1.3_all_chromosomes.zip'\n"
        f"  2. Save it as {REVEL_ZIP} (do NOT commit it -- data/ is gitignored)\n"
        "Then re-run this cell -- it streams the 6.5 GB member CSV and stops as\n"
        "soon as it passes chromosome 7, never loading the whole file into memory."
    )
else:
    start, end = CFTR_WINDOW
    rows, seen7, scanned = [], False, 0
    with zipfile.ZipFile(REVEL_ZIP) as z:
        member_info = z.getinfo(REVEL_MEMBER)
        # The zip's own internal metadata: when REVEL's authors packaged this
        # exact file, straight from the archive -- no download or HTTP call needed.
        packaged_at = datetime(*member_info.date_time).isoformat()
        with z.open(REVEL_MEMBER) as fh:
            reader = csv.reader(io.TextIOWrapper(fh, encoding="utf-8", newline=""))
            header = next(reader)
            tx_idx = header.index("Ensembl_transcriptid")
            for f in reader:
                scanned += 1
                if f[0] == "7":
                    seen7 = True
                    g = f[2]                           # grch38_pos
                    if g in ("", "."):
                        continue
                    p = int(g)
                    if start <= p <= end:
                        rows.append((f[0], p, f[3], f[4], f[5], f[6], round(float(f[7]), 4), f[tx_idx]))
                elif seen7:
                    break                               # left the contiguous chr7 block
    print(f"scanned {scanned:,} REVEL rows to reach/pass chr7")
    print(f"revel_with_transcript_ids packaged at (zip-embedded timestamp): {packaged_at}")
    df = pd.DataFrame(rows, columns=["chrom", "pos", "ref", "alt", "aaref", "aaalt",
                                      "revel_score", "ensembl_transcriptid"])
    df = df.sort_values("pos").reset_index(drop=True)
    df["source"] = "REAL"
    df.to_csv(REVEL_TSV, index=False)
    REVEL_RELEASE_JSON.write_text(json.dumps({
        "zip_member": REVEL_MEMBER,
        "zip_member_packaged_at": packaged_at,
        "note": "packaged_at is the zip's own embedded per-file timestamp (zipfile.ZipInfo.date_time), "
                "not a download date -- it is REVEL's own record of when this file was built.",
    }, indent=2))
    print(f"REAL REVEL CFTR variants written (raw, all transcripts): {len(df):,} "
          f"-> {REVEL_TSV.relative_to(DATA_DIR.parent)}")
    print(f"wrote {REVEL_RELEASE_JSON.relative_to(DATA_DIR.parent)}")


already built -> revel_cftr_v1.3.csv (delete it and revel_cftr_v1.3.release.json to rebuild)


## Load the REAL REVEL data

`tk.load_revel()` reads the extract the cell above just built (or skipped, if it
already existed). The raw CSV has **10,826** rows for CFTR because REVEL lists
one row per genomic site **per distinct amino-acid call**, and CFTR has several
annotated transcripts. `load_revel()` keeps **only** rows tagged with CFTR's
canonical transcript (`ENST00000003084`) — see the "Canonical-transcript
filtering" note above for why the other 1,096 rows (all belonging to one
low-confidence, unconfirmed-start transcript, `ENST00000468795`) are dropped
rather than merged in. That leaves **9,730** unique genomic sites below.

The columns:

| column | meaning |
|--------|---------|
| `chrom` | chromosome (`'7'` for all rows here) |
| `pos` | GRCh38 genomic position |
| `ref` / `alt` | reference/alternate DNA base at that position |
| `aaref` / `aaalt` | REVEL's own reference/alternate amino-acid codes for this change |
| `revel_score` | the 0–1 ensemble score — the star of this notebook |
| `ensembl_transcriptid` | which CFTR transcript(s) this row's amino-acid call applies to (always includes the canonical one, post-filter) |
| `revel_release` | the zip-embedded timestamp from the build cell above |
| `source` | `REAL` — genuine REVEL v1.3 data |


In [3]:
revel = tk.load_revel()    # REAL — genome-wide REVEL v1.3 for CFTR (~9,730), canonical-transcript-only, built above
print(f"{len(revel):,} REAL REVEL variants | source: {revel['source'].unique().tolist()}")
print('columns:', list(revel.columns))
print('revel_score range:', revel['revel_score'].min(), '->', revel['revel_score'].max())
print('revel_release (zip-embedded timestamp):', revel['revel_release'].iloc[0])
print('all rows canonical-transcript-tagged:', revel['ensembl_transcriptid'].str.contains('ENST00000003084').all())
revel.head(6)


9,730 REAL REVEL variants | source: ['REAL']
columns: ['chrom', 'pos', 'ref', 'alt', 'aaref', 'aaalt', 'revel_score', 'ensembl_transcriptid', 'source', 'revel_release']
revel_score range: 0.103 -> 0.996
revel_release (zip-embedded timestamp): 2021-05-03T15:50:38
all rows canonical-transcript-tagged: True


,chrom,pos,ref,alt,aaref,aaalt,revel_score,ensembl_transcriptid,source,revel_release
0,7,117480095,A,C,M,L,0.610,ENST00000003084;ENST00000454343;ENST00000426809,REAL,2021-05-03T15:50:38
1,7,117480095,A,G,M,V,0.636,ENST00000003084;ENST00000454343;ENST00000426809,REAL,2021-05-03T15:50:38
2,7,117480095,A,T,M,L,0.610,ENST00000003084;ENST00000454343;ENST00000426809,REAL,2021-05-03T15:50:38
3,7,117480096,T,A,M,K,0.606,ENST00000003084;ENST00000454343;ENST00000426809,REAL,2021-05-03T15:50:38
4,7,117480096,T,C,M,T,0.669,ENST00000003084;ENST00000454343;ENST00000426809,REAL,2021-05-03T15:50:38
5,7,117480096,T,G,M,R,0.559,ENST00000003084;ENST00000454343;ENST00000426809,REAL,2021-05-03T15:50:38


### Is 9,730 really the ceiling? Check it, don't just trust it

Rather than take that number on faith, let's compute it independently: fetch CFTR's
real canonical CDS sequence live from Ensembl (`ENST00000003084`), and count, codon
by codon, every single-nucleotide substitution that actually changes the amino acid
(excluding synonymous changes and premature stops). If REVEL's canonical table is
truly a complete, saturated set of SNV-missense variants, this independently-computed
count should match `len(revel)` exactly.


In [4]:
import requests

STANDARD_CODE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K','GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGT':'S','AGC':'S','AGA':'R','AGG':'R','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}
BASES = "ACGT"

resp = requests.get(
    "https://rest.ensembl.org/sequence/id/ENST00000003084",
    params={"type": "cds", "content-type": "text/x-fasta"}, timeout=60)
resp.raise_for_status()
cds = "".join(resp.text.splitlines()[1:]).upper()   # drop the FASTA header line
n_codons = len(cds) // 3
print(f"canonical CDS (live from Ensembl): {len(cds):,} nt = {n_codons:,} codons (1,480 aa + stop)")

missense_ceiling = 0
for i in range(n_codons):
    codon = cds[i*3:i*3 + 3]
    wt_aa = STANDARD_CODE.get(codon)
    if wt_aa is None or wt_aa == "*":
        continue                                        # skip the stop codon itself
    for pos in range(3):
        for b in BASES:
            if b == codon[pos]:
                continue
            new_aa = STANDARD_CODE[codon[:pos] + b + codon[pos + 1:]]
            if new_aa not in (wt_aa, "*"):               # missense: changed, and not to a stop
                missense_ceiling += 1

print(f"true SNV-missense ceiling, computed from the real CDS: {missense_ceiling:,}")
print(f"REVEL's canonical row count:                           {len(revel):,}")
print("MATCH -- REVEL achieves full SNV-missense saturation for canonical CFTR"
      if missense_ceiling == len(revel) else "MISMATCH -- investigate")


canonical CDS (live from Ensembl): 4,443 nt = 1,481 codons (1,480 aa + stop)
true SNV-missense ceiling, computed from the real CDS: 9,730
REVEL's canonical row count:                           9,730
MATCH -- REVEL achieves full SNV-missense saturation for canonical CFTR


## 2 · REVEL isn't one cut-point — it's a *graded* scale

A very common shortcut is: *"REVEL ≥ 0.75 → likely pathogenic, otherwise not."* That single cut
throws away information.

The ClinGen **Sequence Variant Interpretation** working group (**Pejaver et al. 2022**,
*AJHG*, **PMID 36413997**) *calibrated* REVEL against the **ACMG/AMP** evidence framework. They
showed a REVEL score maps to **graded tiers of pathogenic evidence strength** (Supporting →
Moderate → Strong), not a single yes/no line:

| REVEL score ≥ | ACMG pathogenic evidence strength (approx.) |
|---|---|
| **0.932** | **Strong** (PP3_Strong) |
| **0.773** | **Moderate** (PP3_Moderate) |
| **0.644** | **Supporting** (PP3_Supporting) |
| **0.290** | *below this* → **Benign** supporting evidence (BP4) |

(These break-points are approximate and are the ones used in the toolkit's teaching table.)

**Why it matters:** the toolkit's single binary cut sits at **0.75** (`tk.THRESHOLDS['revel']`,
below). Two variants at REVEL **0.80** and **0.95** both clear that single line, so a binary
reading calls them the *same thing* — "pathogenic." But the graded calibration says one is only
**Moderate** evidence (0.80, below the 0.932 Strong bar) and the other is **Strong** (0.95).
Collapsing them to one label loses exactly the nuance a curator needs.

The toolkit deliberately ships a **single** binary cut (`tk.THRESHOLDS['revel']`) to keep the
`call_from_score` helper simple — but you should know the richer, graded reality exists. Sorting
the real CFTR REVEL scores into the four tiers makes the spread concrete:


In [5]:
# The toolkit's SIMPLE binary cut-points (one 'pathogenic' line, one 'benign' line):
tk.THRESHOLDS['revel']


{'path': 0.75, 'benign': 0.29}

In [6]:
tier_edges  = [-np.inf, 0.290, 0.644, 0.773, 0.932, np.inf]
tier_labels = ['Benign-supporting (<0.290)', 'Indeterminate (0.290-0.644)',
               'Path Supporting (0.644-0.773)', 'Path Moderate (0.773-0.932)',
               'Path Strong (>=0.932)']
revel['revel_tier'] = pd.cut(revel['revel_score'], bins=tier_edges, labels=tier_labels, right=False)
print('REAL REVEL CFTR variants per Pejaver 2022 evidence tier:\n')
print(revel['revel_tier'].value_counts().reindex(tier_labels).fillna(0).astype(int).to_string())


REAL REVEL CFTR variants per Pejaver 2022 evidence tier:

revel_tier
Benign-supporting (<0.290)        715
Indeterminate (0.290-0.644)      4114
Path Supporting (0.644-0.773)    1731
Path Moderate (0.773-0.932)      2356
Path Strong (>=0.932)             814


Even across this real, unfiltered CFTR set you can see variants spread across **several**
evidence strengths — the detail a single 0.75 cut-point would flatten into just "pathogenic vs
not."


## Example: the shared missense worked-example panel, scored by **REVEL**

The same fixed panel of famous CFTR **missense** variants runs through every missense tool
(tools/01–06, benchmark/00–01), so you can follow one set of variants across the series. The
variant list is `tk.A1_PANEL_VARIANTS` / `tk.A2_KNOWN_CDNA` (shared in `toolkit.py`); the
**scoring is shown inline below** so you can see exactly how REVEL is joined onto it.


In [7]:
# REVEL is COORDINATE-keyed (no protein position in the file). To score the panel we
# bridge each protein key -> genomic coordinate via gnomAD, then look REVEL up by
# coordinate. This IS the join-key lesson, made concrete.
panel = tk.A1_PANEL_VARIANTS
coord = tk.load_gnomad_missense().drop_duplicates('protein_variant').set_index('protein_variant')['variant_id']
rev = tk.load_revel()
rows = []
for pv in panel:
    vid = coord.get(pv)
    score = None
    if isinstance(vid, str) and vid.count('-') == 3:
        _c, p, ref, alt = vid.split('-')
        hit = rev[(rev['pos'] == int(p)) & (rev['ref'] == ref) & (rev['alt'] == alt)]
        score = round(float(hit['revel_score'].iloc[0]), 4) if len(hit) else None
    rows.append({'protein_variant': pv, 'gnomad_coord': vid, 'revel_score': score})
pd.DataFrame(rows)


,protein_variant,gnomad_coord,revel_score
0,G551D,7-117587806-G-A,0.990
1,F508del,None,NaN
2,R117H,7-117530975-G-A,0.807
3,R334W,7-117540230-C-T,0.816
4,G85E,7-117509123-G-A,0.940
5,D1152H,7-117614699-G-C,0.657
6,R668C,7-117592169-C-T,0.706
7,Y161C,7-117531107-A-G,0.969
8,G970D,7-117606674-G-A,0.985
9,S912L,7-117603609-C-T,0.543


## Key takeaways

1. **REVEL** is a **supervised** random-forest ensemble of 13 predictors, trained on curated
   pathogenic/benign variants. Score `[0,1]`, **higher = worse**. Now **REAL** — genome-wide
   REVEL v1.3 for CFTR, 10,826 raw rows collapsing to **9,730** sites scored under CFTR's
   canonical transcript; `source == REAL`.
2. **Coordinate-keyed, not protein-keyed** — REVEL has no protein position, so every join here
   goes through `chrom,pos,ref,alt` (bridged via gnomAD for the worked-example panel above).
   CFTR's plus strand means `ref`/`alt` match the coding change directly; the real gotcha is
   picking `grch38_pos` over `hg19_pos` in the raw file.
3. **Multiple transcripts, one kept.** REVEL's raw file tags rows by which CFTR transcript(s)
   share that amino-acid call. One transcript in this data, `ENST00000468795`, never agrees with
   canonical and doesn't match live Ensembl VEP either (0/1,096) — a stale, low-confidence
   annotation, not real alternative biology. `load_revel()` keeps only canonical-transcript rows.
4. **Circularity stays real:** REVEL's training labels share lineage with ClinVar/HGMD, so a
   REVEL-vs-ClinVar comparison is **partly circular**, unlike the unsupervised tools in
   tools/02–04.
5. **REVEL is a graded scale, not one cut.** The toolkit's single 0.75 line is a simplification;
   Pejaver et al. 2022's calibration splits pathogenic evidence into Supporting/Moderate/Strong
   tiers, and real CFTR scores spread across all of them.
6. **Version-tracked despite the manual download**: the build cell reads the release zip's own
   embedded timestamp (`revel_release` = 2021-05-03T15:50:38) rather than leaving the release
   undated.
7. Non-commercial license — cite REVEL; raw table kept external.

